#Pipeline de Enriquecimento de Dados

###Instalações necessárias

In [ ]:
!pip install prefect loguru rapidfuzz tqdm requests --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55

##Importação das bibliotecas
Nesta etapa importamos as bibliotecas necessárias para:
---
-Consumo de API (request)

---

-Manipulação de dados (pandas)

---
-Similaridade Textual (RapidFuzz)

---
-Orquestração do Pipeline(prefect)

---
-Logging Estruturado (Loguru)

---
Entre outras

In [ ]:
import os
import sys
import time
import requests
import pandas as pd
from rapidfuzz import fuzz
from tqdm import tqdm
from loguru import logger
from prefect import flow, task

###Configurações de Ambiente

---


*   Chave da API do TMDB
*   URL base da API
*   Delay entre requisições(para evitar bloqueios )
* Thresold de similaridade Textual
*  Estrutura de cache para reduzir chamadas repetidas

---
Essas configurações garatem estabilidade e controle do consumo da API



In [ ]:
API_KEY = ""
BASE_URL = "https://api.themoviedb.org/3"

REQUEST_DELAY = 0.25
FUZZY_THRESHOLD = 85

os.makedirs("logs", exist_ok=True)

logger.remove()

logger.add(
    sys.stdout,
    level="INFO",
    colorize=True,
    format=(
        "<green>{time:YYYY-MM-DD HH:mm:ss}</green> | "
        "<level>{level: <8}</level> | "
        "<cyan>{extra[step]}</cyan> {message}"
    ),
)

logger.add(
    "logs/enrichment_{time}.log",
    rotation="10 MB",
    level="INFO",
    format="{time} | {level} | {extra[step]} {message}",
)

2

In [ ]:
id_cache = {}

###Funções:
---
### Normalização de Texto

Função auxiliar para padronizar os títulos antes da comparação.

Remove diferenças causadas por:
- Maiúsculas/minúsculas
- Espaços desnecessários

Isso melhora a precisão do matching.

---
### search:
Esta é a função central do matching.

Ela:
1. Busca o título na API
2. Avalia múltiplos resultados
3. Extrai o ano da data retornada
4. Aplica validação temporal

Critério principal:
Só aceita resultados com diferença de ano ≤ 3.

Isso reduz drasticamente falsos positivos.

---
###get_tmdb_details:

 Coleta de Informações Detalhadas

Após validar o ID no TMDB, esta função coleta:

- Data oficial de lançamento
- Diretor (para filmes)
- Elenco principal (top 5)
- País de produção

A consulta só ocorre após a validação temporal.

---


In [ ]:
def normalize(text):
    return text.lower().strip()

from datetime import datetime

def search_tmdb(title, year, content_type):

    cache_key = f"{title}_{year}_{content_type}"
    if cache_key in id_cache:
        return id_cache[cache_key]

    endpoint = "/search/movie" if content_type == "Movie" else "/search/tv"

    params = {
        "api_key": API_KEY,
        "query": title
    }

    response = requests.get(BASE_URL + endpoint, params=params)

    if response.status_code != 200:
        return None

    results = response.json().get("results", [])

    if not results:
        return None

    current_year = datetime.now().year

    best_match = None

    for result in results:

        api_title = result.get("title") or result.get("name")
        original_title = result.get("original_title") or result.get("original_name")

        api_date = (
            result.get("release_date") or
            result.get("first_air_date") or ""
        )

        if not api_date or len(api_date) < 4:
            continue

        api_year = int(api_date[:4])

        # Regras de segurança temporal
        if api_year < 1920:
            continue

        if api_year > current_year:
            continue

        year_diff = abs(api_year - int(year))

        #  Regra principal: diferença máxima de 3 anos
        if year_diff > 3:
            continue

        # 1️ Strict Match (ano coerente)
        if year_diff <= 1:
            id_cache[cache_key] = result["id"]
            return result["id"]

        # 2️ Original title match
        if original_title and normalize(original_title) == normalize(title):
            id_cache[cache_key] = result["id"]
            return result["id"]

        # 3️ Fuzzy match controlado
        if fuzz.ratio(normalize(api_title), normalize(title)) > FUZZY_THRESHOLD:
            best_match = result["id"]

    if best_match:
        id_cache[cache_key] = best_match
        return best_match

    return None

def get_tmdb_details(tmdb_id, content_type):

    endpoint = f"/movie/{tmdb_id}" if content_type == "Movie" else f"/tv/{tmdb_id}"

    params = {
        "api_key": API_KEY,
        "append_to_response": "credits"
    }

    response = requests.get(BASE_URL + endpoint, params=params)

    if response.status_code != 200:
        return None

    data = response.json()

    # Release date
    if content_type == "Movie":
        release_date = data.get("release_date")
    else:
        release_date = data.get("first_air_date")

    # Country
    if content_type == "Movie":
        countries = data.get("production_countries", [])
        country = ", ".join([c["name"] for c in countries]) if countries else None
    else:
        countries = data.get("origin_country", [])
        country = ", ".join(countries) if countries else None

    # Director (somente Movie)
    director = None
    if content_type == "Movie":
        for crew in data.get("credits", {}).get("crew", []):
            if crew.get("job") == "Director":
                director = crew.get("name")
                break

    # Cast (top 5)
    cast_list = data.get("credits", {}).get("cast", [])[:5]
    cast = ", ".join([c["name"] for c in cast_list]) if cast_list else None

    return {
        "release_date_api": release_date,
        "director_api": director,
        "cast_api": cast,
        "country_api": country
    }
def get_release_date(tmdb_id, content_type):

    endpoint = f"/movie/{tmdb_id}" if content_type == "Movie" else f"/tv/{tmdb_id}"

    params = {
        "api_key": API_KEY
    }

    response = requests.get(BASE_URL + endpoint, params=params)

    if response.status_code != 200:
        return None

    data = response.json()

    if content_type == "Movie":
        return data.get("release_date")

    return data.get("first_air_date")

###Tasks:

---

Carregamento do Dataset

Task responsável por:

- Ler o CSV original
- Retornar o DataFrame para o pipeline

Mantemos a estrutura original intacta.

---
Enriquecimento do Dataset

Nesta etapa:

- Cada título é validado via TMDB
- Caso aprovado, coletamos os dados adicionais
- Caso não aprovado, registramos como "nao_informado"

Essa abordagem prioriza precisão em vez de preenchimento forçado.

---
Tratamento Final

Substituímos valores nulos por "nao_informado" para:

- Padronizar o dataset
- Evitar problemas em análises posteriores
- Tornar o preenchimento explícito

---



In [ ]:

@task
def load_dataset(path):
    logger.bind(step="load").info("Carregando dataset...")
    df = pd.read_csv(path)
    return df
@task
def enrich_release_date(df):

    logger.bind(step="enrich").info("Iniciando enriquecimento...")

    release_dates = []
    director_list = []
    cast_list = []
    country_list = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        title = row["title"]
        year = row["release_year"]
        content_type = row["type"]

        tmdb_id = search_tmdb(title, year, content_type)

        if tmdb_id:
            details = get_tmdb_details(tmdb_id, content_type)
        else:
            details = None

        if details:
            release_dates.append(details["release_date_api"])
            director_list.append(details["director_api"])
            cast_list.append(details["cast_api"])
            country_list.append(details["country_api"])
        else:
            release_dates.append(None)
            director_list.append(None)
            cast_list.append(None)
            country_list.append(None)

        time.sleep(REQUEST_DELAY)

    df["release_date_api"] = release_dates
    df["director_api"] = director_list
    df["cast_api"] = cast_list
    df["country_api"] = country_list

    logger.bind(step="enrich").info("Enriquecimento finalizado.")

    return df


@task
def finalize_dataset(df):

    columns_to_fill = [
        "release_date_api",
        "director_api",
        "cast_api",
        "country_api"
    ]

    for col in columns_to_fill:
        df[col] = df[col].fillna("nao_informado")

    logger.bind(step="finalize").info("Tratamento final aplicado.")

    return df


@task
def save_dataset(df, path):

    df.to_csv(path, index=False)
    logger.bind(step="save").success(f"Dataset salvo em {path}")

###Flow:

---
Orquestração do Pipeline

O Flow coordena todas as etapas:

1. Carregamento
2. Enriquecimento
3. Tratamento final
4. Salvamento

Isso garante execução organizada e rastreável.

In [ ]:
@flow(name="netflix-enrichment-pro")
def enrichment_pipeline(input_path, output_path):

    df = load_dataset(input_path)
    df = enrich_release_date(df)
    df = finalize_dataset(df)
    save_dataset(df, output_path)

###Execução

In [ ]:
enrichment_pipeline(
    input_path="netflix_titles.csv",
    output_path="df_netflix_enriquecido.csv"
)

INFO:prefect.flow_runs:Beginning flow run 'winged-husky' for flow 'netflix-enrichment-pro'


2026-03-06 22:26:44 | INFO     | load Carregando dataset...


INFO:prefect.task_runs:Finished in state Completed()


2026-03-06 22:26:44 | INFO     | enrich Iniciando enriquecimento...


100%|██████████| 8807/8807 [1:08:21<00:00,  2.15it/s]

...


INFO:prefect.task_runs:Finished in state Completed()


2026-03-06 23:35:05 | INFO     | finalize Tratamento final aplicado.


INFO:prefect.task_runs:Finished in state Completed()


2026-03-06 23:35:06 | SUCCESS  | save Dataset salvo em df_netflix_enriquecido.csv


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.flow_runs:Finished in state Completed()


###Testes:

---
Validação e Controle de Qualidade

Aqui realizamos verificações como:

- Percentual de preenchimento
- Diferença entre anos
- Distribuição dos resultados

Essa etapa garante confiabilidade estatística do enriquecimento.

In [ ]:
df = pd.read_csv("df_netflix_enriquecido.csv")

print((df["release_date_api"] != "nao_informado").mean())
print(df["release_date_api"].value_counts().head())

0.9320994663335983
release_date_api
nao_informado    598
2019-11-01        15
2018-09-21        15
2019-01-18        15
2017-10-13        14
Name: count, dtype: int64


In [ ]:
# garantir que só datas válidas sejam usadas, tem que retornar ate 3
df_valid = df[
    (df["release_date_api"] != "nao_informado") &
    (df["release_date_api"].notna())
].copy()

df_valid["api_year"] = pd.to_numeric(
    df_valid["release_date_api"].str[:4],
    errors="coerce"
)

df_valid = df_valid[df_valid["api_year"].notna()]

df_valid["year_diff"] = abs(
    df_valid["api_year"] - df_valid["release_year"]
)

df_valid["year_diff"].value_counts()

,count
year_diff,
0,6627
1,1024
2,343
3,215


In [ ]:
df.groupby("type")["release_date_api"].count()

,release_date_api
type,
Movie,6131
TV Show,2676


In [ ]:
df_valid["api_year"].value_counts().sort_index().tail(30)

,count
api_year,
1995,24
1996,24
1997,32
1998,31
1999,32
2000,32
2001,44
2002,51
2003,60


In [ ]:
print("Linhas:", len(df))

Linhas: 8807


In [ ]:
colunas_originais = [
    "show_id","type","title","director","cast",
    "country","date_added","release_year",
    "rating","duration","listed_in","description"
]

for col in colunas_originais:
    if col not in df.columns:
        print("Coluna perdida:", col)

In [ ]:
novas_colunas = [
    "release_date_api",
    "director_api",
    "cast_api",
    "country_api"
]

for col in novas_colunas:
    preenchido = (df[col] != "nao_informado").mean()
    print(f"{col}: {round(preenchido*100,2)}% preenchido")

release_date_api: 93.21% preenchido
director_api: 66.07% preenchido
cast_api: 87.08% preenchido
country_api: 87.26% preenchido


In [ ]:
df_valid = df[df["release_date_api"] != "nao_informado"].copy()

df_valid["api_year"] = pd.to_numeric(
    df_valid["release_date_api"].str[:4],
    errors="coerce"
)

df_valid["year_diff"] = abs(
    df_valid["api_year"] - df_valid["release_year"]
)

print(df_valid["year_diff"].value_counts().sort_index())

year_diff
0    6627
1    1024
2     343
3     215
Name: count, dtype: int64


In [ ]:
df[df["type"] == "TV Show"]["director_api"].value_counts().head()

,count
director_api,
nao_informado,2676


In [ ]:
df.sample(5)[
    ["title","type","release_year",
     "release_date_api",
     "director_api",
     "country_api"]
]

,title,type,release_year,release_date_api,director_api,country_api
3138,Holy Expectations,Movie,2019,2019-09-05,Fernando Ayllón,Colombia
8161,Tees Maar Khan,Movie,2010,2010-12-24,Farah Khan,India
7205,Kills on Wheels,Movie,2016,2016-04-28,Attila Till,Hungary
689,Mobile Suit Gundam I,Movie,1981,1981-03-14,Ryoji Fujiwara,Japan
5893,Winter on Fire: Ukraine's Fight for Freedom,Movie,2015,2015-09-03,Evgeny Afineevsky,"United Kingdom, United States of America, Ukraine"


In [ ]:
df["country_api"].value_counts().head(10)

,count
country_api,
United States of America,2007
nao_informado,1122
India,753
US,725
JP,208
GB,189
United Kingdom,184
KR,176
Canada,109


In [ ]:
(df["cast_api"] == "nao_informado").mean()

np.float64(0.12921539684342)

In [ ]:
df[df["title"].str.contains("Inception", case=False)][
    ["title","cast_api","director_api"]
]  #teste procurando filme conhecido deve retornar diretor Christopher Nolan

,title,cast_api,director_api
340,Inception,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",Christopher Nolan


In [ ]:
df_valid[df_valid["year_diff"] == 3][
    ["title","release_year","release_date_api"]
].sample(10)  #validação entre diferenças de anos, aceito pela análise até 3 anos

,title,release_year,release_date_api
2060,Always Be My Maybe,2016,2019-05-31
2611,Tjovitjo,2017,2020-08-20
7698,Pajanimals,2011,2008-11-02
2546,Bordertown,2019,2016-01-03
5138,All Hail King Julien,2017,2014-12-19
4363,Minecraft: Story Mode,2015,2018-11-27
6233,Barbecue,2017,2014-04-30
4028,Into the Badlands,2018,2015-11-15
8646,Twirlywoos,2018,2015-02-23
2043,Villain,2020,2023-03-25
